# GSTBuddy — Baseline Evaluation (Week 2)

Evaluate an untouched small instruct model on the GST tool-calling test set **before fine-tuning**.

**Base model:** `Qwen/Qwen3-1.7B`  
**Evaluation:** first assistant tool-call turn (important: sequential examples may require tool execution before the next call).  
**Metrics:** tool-call decision accuracy, tool-name accuracy, argument exact-match accuracy, valid tool-call JSON rate, and no-tool accuracy.

> This is a baseline harness, not a tax-accuracy audit. It measures whether the model selects and formats the expected tool call.

## 1. Enable a GPU

In Colab, select **Runtime → Change runtime type → T4 GPU** (if available), then run this cell.

In [ ]:
!nvidia-smi

## 2. Install dependencies

This notebook uses Transformers for inference. The 1.7B model is loaded in FP16; no quantization is needed for this baseline.

In [ ]:
!pip -q install -U "transformers>=4.51.0" accelerate pandas

## 3. Upload the Week 1 files

Upload:
- `test.jsonl`
- `tools.json`

You do not need `train.jsonl` for baseline evaluation.

In [ ]:
from google.colab import files
uploaded = files.upload()

required = {"test.jsonl", "tools.json"}
missing = required - set(uploaded.keys())
if missing:
    raise FileNotFoundError(f"Please upload these files: {sorted(missing)}")

TEST_PATH = "test.jsonl"
TOOLS_PATH = "tools.json"

## 4. Load and inspect the test set

In [ ]:
import json
from pathlib import Path
from collections import Counter

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

with open(TOOLS_PATH, "r", encoding="utf-8") as f:
    TOOLS = json.load(f)

print("Test examples:", len(test_rows))
print("Tool schemas:", len(TOOLS))
print("Categories:", Counter(row["category"] for row in test_rows))
print("Languages:", Counter(row["lang"] for row in test_rows))
print("\nExample user prompt:")
print(next(m["content"] for m in test_rows[0]["messages"] if m["role"] == "user"))

## 5. Load the untouched base model

We use Qwen3-1.7B as a compact baseline. Keep this model ID unchanged when comparing against the fine-tuned adapter later.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-1.7B"

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. In Colab choose Runtime → Change runtime type → GPU.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

print("Loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))

## 6. Prepare prompts and parse tool calls

The source dataset stores OpenAI-style tool-call messages. Qwen3's chat template renders the available tool definitions into its native tool-call prompt format.

For multi-step examples, we score only the **first assistant tool-call turn**. This avoids incorrectly expecting a model to call a dependent second tool before it has received the first tool's result.

In [ ]:
import re
import math

def get_user_and_system(row):
    system_msg = next((m for m in row["messages"] if m["role"] == "system"), None)
    user_msg = next(m for m in row["messages"] if m["role"] == "user")
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg["content"]})
    messages.append({"role": "user", "content": user_msg["content"]})
    return messages

def expected_first_turn_calls(row):
    for message in row["messages"]:
        if message["role"] == "assistant" and message.get("tool_calls"):
            calls = []
            for tc in message["tool_calls"]:
                calls.append({
                    "name": tc["function"]["name"],
                    "arguments": json.loads(tc["function"]["arguments"])
                })
            return calls
    return []

def parse_qwen_tool_calls(text):
    # Qwen tool-call format: <tool_call>{"name": "...", "arguments": {...}}</tool_call>
    blocks = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, flags=re.DOTALL)
    parsed = []
    errors = []
    for block in blocks:
        try:
            obj = json.loads(block.strip())
            if "name" in obj and "arguments" in obj:
                parsed.append({"name": obj["name"], "arguments": obj["arguments"]})
            else:
                errors.append("JSON missing name/arguments")
        except Exception as exc:
            errors.append(str(exc))
    return parsed, len(blocks), errors

def values_equal(a, b, tol=1e-6):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return math.isclose(float(a), float(b), rel_tol=tol, abs_tol=tol)
    if isinstance(a, dict) and isinstance(b, dict):
        return set(a) == set(b) and all(values_equal(a[k], b[k], tol) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(values_equal(x, y, tol) for x, y in zip(a, b))
    return a == b

## 7. Run baseline inference

Start with 100 examples to confirm everything works. After the smoke test, change `EVAL_LIMIT` to `None` to evaluate all 601 test examples.

The generation is deterministic (`do_sample=False`). The notebook stores raw model output so failures can be inspected.

In [ ]:
EVAL_LIMIT = 100  # Set to None for the complete test set
MAX_NEW_TOKENS = 256

rows_to_eval = test_rows if EVAL_LIMIT is None else test_rows[:EVAL_LIMIT]
predictions = []

for idx, row in enumerate(rows_to_eval, start=1):
    prompt_messages = get_user_and_system(row)

    encoded = tokenizer.apply_chat_template(
        prompt_messages,
        tools=TOOLS,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        enable_thinking=False
    )
    input_ids = encoded.to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = generated[0][input_ids.shape[-1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=False)
    predicted_calls, block_count, parse_errors = parse_qwen_tool_calls(raw_output)
    expected_calls = expected_first_turn_calls(row)

    predictions.append({
        "id": row["id"],
        "category": row["category"],
        "lang": row["lang"],
        "expected_calls": expected_calls,
        "predicted_calls": predicted_calls,
        "tool_blocks_found": block_count,
        "parse_errors": parse_errors,
        "raw_output": raw_output
    })

    if idx % 10 == 0 or idx == len(rows_to_eval):
        print(f"Evaluated {idx}/{len(rows_to_eval)}")

print("Inference complete.")

## 8. Calculate metrics

Definitions:
- **Tool-call decision accuracy:** correct decision to call at least one tool or call no tool.
- **Tool-name exact match:** predicted first-turn tool names and order match the expected names.
- **Argument exact match:** tool names and all arguments match (numeric values allow a tiny floating-point tolerance).
- **Valid JSON rate:** fraction of emitted `<tool_call>` blocks that parse as JSON with `name` and `arguments`.
- **No-tool accuracy:** among examples whose expected first turn has no tool call, fraction where the model emits no tool call.

These metrics are intentionally strict and should be reported with the evaluation sample size.

In [ ]:
n = len(predictions)
decision_correct = 0
tool_name_correct = 0
args_correct = 0
expected_tool_examples = 0
no_tool_examples = 0
no_tool_correct = 0
total_tool_blocks = 0
valid_tool_blocks = 0

for p in predictions:
    exp = p["expected_calls"]
    pred = p["predicted_calls"]

    total_tool_blocks += p["tool_blocks_found"]
    valid_tool_blocks += len(pred)

    if bool(exp) == bool(pred):
        decision_correct += 1

    if exp:
        expected_tool_examples += 1
        exp_names = [x["name"] for x in exp]
        pred_names = [x["name"] for x in pred]
        if exp_names == pred_names:
            tool_name_correct += 1
            if len(exp) == len(pred) and all(
                values_equal(e["arguments"], a["arguments"])
                for e, a in zip(exp, pred)
            ):
                args_correct += 1
    else:
        no_tool_examples += 1
        if not pred:
            no_tool_correct += 1

metrics = {
    "evaluated_examples": n,
    "tool_call_decision_accuracy": decision_correct / n if n else 0,
    "tool_name_exact_match_on_tool_examples": tool_name_correct / expected_tool_examples if expected_tool_examples else 0,
    "tool_and_arguments_exact_match_on_tool_examples": args_correct / expected_tool_examples if expected_tool_examples else 0,
    "valid_tool_call_json_rate": valid_tool_blocks / total_tool_blocks if total_tool_blocks else 0,
    "no_tool_accuracy": no_tool_correct / no_tool_examples if no_tool_examples else 0,
    "tool_examples": expected_tool_examples,
    "no_tool_examples": no_tool_examples,
    "tool_call_blocks_emitted": total_tool_blocks
}

for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

## 9. Save results

Download the CSV and JSON metrics. Keep these files: after fine-tuning, run the same evaluation notebook with the adapter loaded and compare the results.

In [ ]:
import pandas as pd

report_rows = []
for p in predictions:
    report_rows.append({
        "id": p["id"],
        "category": p["category"],
        "lang": p["lang"],
        "expected_calls": json.dumps(p["expected_calls"], ensure_ascii=False),
        "predicted_calls": json.dumps(p["predicted_calls"], ensure_ascii=False),
        "parse_errors": json.dumps(p["parse_errors"], ensure_ascii=False),
        "raw_output": p["raw_output"]
    })

pd.DataFrame(report_rows).to_csv("baseline_predictions.csv", index=False)
with open("baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Saved baseline_predictions.csv and baseline_metrics.json")

from google.colab import files
files.download("baseline_predictions.csv")
files.download("baseline_metrics.json")

## 10. Inspect failures

Look at examples where the model selected the wrong tool, missed a tool call, or produced invalid tool-call JSON. These are useful for deciding what to improve in the training data.

In [ ]:
for p in predictions:
    exp = p["expected_calls"]
    pred = p["predicted_calls"]
    if bool(exp) != bool(pred) or [x["name"] for x in exp] != [x["name"] for x in pred]:
        print("\n---", p["id"], p["category"], p["lang"], "---")
        print("EXPECTED:", exp)
        print("PREDICTED:", pred)
        print("RAW OUTPUT:", p["raw_output"][:1200])
        break